In [4]:
from src.data_loader import DataLoader
from src.data_analyzer import DataAnalyzer

loader = DataLoader("../data/raw/bank-full.csv")
df = loader.load_data()

analyzer = DataAnalyzer(df)

print(f"Dataset shape: {df.shape}")

Dataset shape: (45211, 17)


In [5]:
df.dtypes

age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object

In [6]:
df_model = df.copy()

df_model = df_model.drop(columns=["duration"])

print("Dataset shape:", df_model.shape)
print("\nColumns:")
print(df_model.columns.tolist())

Dataset shape: (45211, 16)

Columns:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous', 'poutcome', 'y']


In [7]:
X = df_model.drop(columns=["y"])
y = df_model["y"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (45211, 15)
y shape: (45211,)

Target distribution:
y
no     39922
yes     5289
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

X_train: (36168, 15)
X_test: (9043, 15)
y_train: (36168,)
y_test: (9043,)

Train target distribution:
y
no     88.3
yes    11.7
Name: proportion, dtype: float64

Test target distribution:
y
no     88.3
yes    11.7
Name: proportion, dtype: float64


In [9]:
categorical_columns = X_train.select_dtypes(
    include=["str", "object", "category"]
).columns.tolist()

numerical_columns = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

print("\nNumerical columns:")
print(numerical_columns)

print("\nNumber of categorical columns:", len(categorical_columns))
print("Number of numerical columns:", len(numerical_columns))

Categorical columns:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

Numerical columns:
['age', 'balance', 'day', 'campaign', 'pdays', 'previous']

Number of categorical columns: 9
Number of numerical columns: 6


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_columns
        ),
        (
            "numerical",
            "passthrough",
            numerical_columns
        )
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [11]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)

X_train_processed shape: (36168, 50)
X_test_processed shape: (9043, 50)


In [12]:
feature_names = preprocessor.get_feature_names_out()

print("Number of features:", len(feature_names))
print("\nFeature names:")
for feature in feature_names:
    print(feature)

Number of features: 50

Feature names:
categorical__job_admin.
categorical__job_blue-collar
categorical__job_entrepreneur
categorical__job_housemaid
categorical__job_management
categorical__job_retired
categorical__job_self-employed
categorical__job_services
categorical__job_student
categorical__job_technician
categorical__job_unemployed
categorical__job_unknown
categorical__marital_divorced
categorical__marital_married
categorical__marital_single
categorical__education_primary
categorical__education_secondary
categorical__education_tertiary
categorical__education_unknown
categorical__default_no
categorical__default_yes
categorical__housing_no
categorical__housing_yes
categorical__loan_no
categorical__loan_yes
categorical__contact_cellular
categorical__contact_telephone
categorical__contact_unknown
categorical__month_apr
categorical__month_aug
categorical__month_dec
categorical__month_feb
categorical__month_jan
categorical__month_jul
categorical__month_jun
categorical__month_mar
catego

In [13]:
y_train_encoded = y_train.map({"no": 0, "yes": 1})
y_test_encoded = y_test.map({"no": 0, "yes": 1})

print("Train target:")
print(y_train_encoded.value_counts())

print("\nTest target:")
print(y_test_encoded.value_counts())

print("\nMissing values in y_train:", y_train_encoded.isna().sum())
print("Missing values in y_test:", y_test_encoded.isna().sum())

Train target:
y
0    31937
1     4231
Name: count, dtype: int64

Test target:
y
0    7985
1    1058
Name: count, dtype: int64

Missing values in y_train: 0
Missing values in y_test: 0


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

logistic_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_columns
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_columns
        )
    ]
)

print("Preprocessor created successfully.")